# My two-model bulk collection

I collect the frozen 1,000 shared puzzles using my unchanged pilot prompts and settings. Each provider must independently pass eight capability checks, complete the 20-training-case operational prefix and verify immutable resume before advancing to the rest of the sample. An unavailable provider does not stop the other provider.

I use `RUN_MODE="bulk"` and `WAIT_FOR_QUOTA=False` to collect until each provider completes or pauses, then report its limit. I can rerun the same notebook later to resume cached answers. If I enable automatic waiting, my computer and kernel must remain running. I make no paid-plan changes automatically.

I use notebook 05 for a separately labeled, training-only exploratory checkpoint and notebook 06 for the unchanged final analysis. I leave held-out predictions and evaluation closed until the final analysis gates pass. This operational revision preserves all prior answers, the sample, model settings, scoring and the final analysis plan; its predecessor is archived with verified hashes.

In [1]:
RUN_MODE = "bulk"  # setup, warmup, or bulk
WAIT_FOR_QUOTA = False  # I stop and report when either provider reaches its limit


## 1. I verify my frozen settings and scoring

In [2]:

from pathlib import Path
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
import os, json, hashlib, re, time, math, uuid, warnings, platform
import requests, chess, nbformat
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'experiments/stage_index.json').exists())
CFG = ROOT/'experiments/bulk_v1'
OUT = ROOT/'data/experiments/bulk_v1'; OUT.mkdir(exist_ok=True)
CONFIG = json.loads((CFG/'config.json').read_text())
PLAN = json.loads((CFG/'analysis_plan.json').read_text())
PROMPT = (CFG/'prompt.txt').read_text()
assert RUN_MODE in ['setup', 'warmup', 'bulk'] and isinstance(WAIT_FOR_QUOTA, bool)
PACIFIC = ZoneInfo('America/Los_Angeles')
SECRETS = [(ROOT/s['key_file']).read_text(encoding='utf-8-sig').strip() for s in CONFIG['models'].values()]
assert all(s and len(s.splitlines()) == 1 for s in SECRETS)

def sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def redact(value):
    if isinstance(value, str):
        for secret in SECRETS: value = value.replace(secret, '[REDACTED]')
        return value
    if isinstance(value, dict): return {k: redact(v) for k, v in value.items()}
    if isinstance(value, list): return [redact(v) for v in value]
    return value

def save(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix+'.tmp')
    temporary.write_text(json.dumps(redact(value), indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
    os.replace(temporary, path)

def records(path):
    path = Path(path)
    if not path.exists(): return []
    text = path.read_text(encoding='utf-8'); lines = text.splitlines(); rows = []
    for i, line in enumerate(lines):
        if not line.strip(): continue
        try: rows.append(json.loads(line))
        except json.JSONDecodeError:
            if i == len(lines)-1 and not text.endswith('\n'):
                warnings.warn(f'I found an incomplete final log record in {path.name}; it is not a completed response.')
            else: raise
    return rows

def append(path, row):
    path = Path(path)
    # I repair only an incomplete last line after a crash, under the provider file lock.
    if path.exists():
        raw = path.read_bytes()
        if raw and not raw.endswith(b'\n'):
            try: json.loads(raw.splitlines()[-1]); prefix = raw+b'\n'
            except json.JSONDecodeError: prefix = raw[:raw.rfind(b'\n')+1]
            path.write_bytes(prefix)
    with path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(redact(row), ensure_ascii=False)+'\n'); stream.flush(); os.fsync(stream.fileno())

@contextmanager
def provider_lock(folder):
    # OS locks are released if the kernel exits, so stale PID files cannot block a resume.
    with (folder/'collection.lock').open('a+b') as handle:
        if handle.tell() == 0: handle.write(b'0'); handle.flush()
        handle.seek(0)
        if os.name == 'nt':
            import msvcrt
            msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
        else:
            import fcntl
            fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        try: yield
        finally:
            handle.seek(0)
            if os.name == 'nt': msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
            else: fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

def code_hash(name):
    n = nbformat.read(ROOT/'notebooks'/name, as_version=4)
    code = '\n\n'.join(c.source for c in n.cells if c.cell_type == 'code' and 'run_control' not in c.metadata.get('tags', []))
    return hashlib.sha256(code.encode()).hexdigest()

manifest = {'version': CONFIG['version'], 'config_sha256': sha(CFG/'config.json'),
            'plan_sha256': sha(CFG/'analysis_plan.json'), 'prompt_sha256': sha(CFG/'prompt.txt'),
            'collection_code_sha256': code_hash('04_bulk_collection.ipynb'),
            'analysis_code_sha256': code_hash('06_final_analysis.ipynb')}
if (OUT/'manifest.json').exists():
    assert manifest == json.loads((OUT/'manifest.json').read_text()), 'Frozen study changed; create a separate version.'
else: save(OUT/'manifest.json', manifest)

# I protect historical raw responses and fitted models. Viewer-facing Markdown is editable.
if not (OUT/'historical_hashes.json').exists():
    protected = {p.relative_to(ROOT).as_posix(): sha(p)
                 for parent in ['data/experiments', 'data/analysis']
                 for p in (ROOT/parent).rglob('*') if p.is_file() and OUT not in p.parents}
    save(OUT/'historical_hashes.json', protected)
else: protected = json.loads((OUT/'historical_hashes.json').read_text())
assert all(sha(ROOT/p) == digest for p, digest in protected.items())
print('I verified my frozen protocol and protected earlier response/model files.')
def score(board,task,response,expected=None):
    choice=response['choices'][0]
    if choice.get('finish_reason')=='length':return 'generation_limit',None,False
    if choice.get('finish_reason')!='stop':return 'nonstop_termination',None,False
    answer=choice.get('message',{}).get('content')
    if not isinstance(answer,str):return 'unparseable',None,False
    answer=answer.strip()
    if task=='side':
        if answer not in ['white','black']:return 'unparseable',answer,False
        return ('success' if answer==('white' if board.turn else 'black') else 'incorrect_side'),answer,True
    if re.fullmatch('[a-h][1-8][a-h][1-8][qrbn]?',answer):
        move=chess.Move.from_uci(answer)
        if move not in board.legal_moves:return 'illegal_move',answer,False
    else:
        try:
            move=board.parse_san(answer)
            if move not in board.legal_moves:return 'illegal_move',answer,False
            canonical=board.san(move)
            normalized=answer.replace('0-0-0','O-O-O').replace('0-0','O-O')
            if normalized not in {canonical,canonical.rstrip('+#')}:return 'unparseable',answer,False
        except ValueError:return 'unparseable',answer,False
    answer=move.uci()
    if task=='mate':
        copy=board.copy();copy.push(move)
        correct=copy.is_checkmate()
    else:correct=task=='legal' or answer==expected
    return ('success' if correct else 'legal_incorrect'),answer,True
def seconds(value):
    try:return float(value)
    except (ValueError,TypeError):
        pairs=re.findall(r'([0-9.]+)(ms|s|m|h)',str(value))
        if pairs:return sum(float(n)*{'ms':.001,'s':1,'m':60,'h':3600}[unit] for n,unit in pairs)
        try:return max(0,(datetime.strptime(value,'%a, %d %b %Y %H:%M:%S GMT').replace(tzinfo=timezone.utc)-datetime.now(timezone.utc)).total_seconds())
        except (ValueError,TypeError):return None

def prompt_for(board, task='evaluation'):
    pieces = '\n'.join(('White' if color else 'Black')+' pieces: '+', '.join(
        f'{chess.piece_name(p.piece_type)} {chess.square_name(s)}'
        for s, p in sorted(board.piece_map().items()) if p.color == color)
        for color in [chess.WHITE, chess.BLACK])
    text = PROMPT.format(side='White' if board.turn else 'Black', fen=board.fen(),
                         pieces=pieces, choices=', '.join(sorted(m.uci() for m in board.legal_moves)))
    if task == 'side': return f'Which side moves next in this FEN? {board.fen()}\n'+pieces+'\nReturn only white or black in the final answer.'
    if task == 'mate': return text.replace('Choose the best next move', 'Choose a move that delivers checkmate immediately')
    if task == 'legal': return text.replace('Choose the best next move', 'Choose any legal next move')
    return text

def request_body(spec, board, task='evaluation'):
    text = prompt_for(board, task)
    if spec['provider'] == 'groq':
        return {'model': spec['model'], 'temperature': spec['temperature'], 'seed': spec['seed'],
                'reasoning_effort': spec['reasoning_effort'], 'max_completion_tokens': spec['max_completion_tokens'],
                'messages': [{'role': 'user', 'content': text}]}
    return {'contents': [{'role': 'user', 'parts': [{'text': text}]}],
            'generationConfig': {'temperature': spec['temperature'], 'topP': spec['top_p'], 'seed': spec['seed'],
                'maxOutputTokens': spec['max_completion_tokens'],
                'thinkingConfig': {'thinkingLevel': spec['thinking_level'], 'includeThoughts': False}}}

def normalized(spec, raw):
    if spec['provider'] == 'groq': return raw
    candidates = raw.get('candidates', [])
    if not candidates: return {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
    candidate = candidates[0]; reason = candidate.get('finishReason', 'UNKNOWN')
    content = ''.join(p.get('text', '') for p in candidate.get('content', {}).get('parts', []) if not p.get('thought', False))
    return {'choices': [{'message': {'content': content}, 'finish_reason': {'STOP': 'stop', 'MAX_TOKENS': 'length'}.get(reason, reason.lower())}]}

start = chess.Board(); after = start.copy(); after.push_uci('e2e4')
mate = chess.Board('7k/8/5KQ1/8/8/8/8/8 w - - 0 1'); mirror = mate.transform(chess.flip_horizontal)
FIXTURES = [('mate_a_white', mate, 'mate'), ('side_white', start, 'side'), ('legal_white', start, 'legal'),
            ('side_black', after, 'side'), ('legal_black', after, 'legal'), ('mate_a_black', mate.mirror(), 'mate'),
            ('mate_b_white', mirror, 'mate'), ('mate_b_black', mirror.mirror(), 'mate')]


I verified my frozen protocol and protected earlier response/model files.


## 2. I freeze my fresh sample and split

In [3]:

def position_key(fen): return ' '.join(str(fen).split()[:4])
def historical_exclusions():
    ids, positions, sources = set(), set(), []
    def add_id(value):
        if isinstance(value, str) and re.fullmatch('[A-Za-z0-9]{5}', value): ids.add(value)
    def scan_metadata(value):
        if isinstance(value, dict):
            for k, v in value.items():
                if k in ['excluded_ids', 'reserved_ids', 'sample_ids', 'puzzle_ids'] and isinstance(v, list):
                    for item in v: add_id(item)
                elif isinstance(v, (dict, list)): scan_metadata(v)
        elif isinstance(value, list):
            for v in value:
                if isinstance(v, (dict, list)): scan_metadata(v)
    for parent in [ROOT/'data/samples', ROOT/'data/experiments']:
        for p in sorted(parent.rglob('*')):
            if not p.is_file() or OUT in p.parents: continue
            before = len(ids)
            if p.suffix == '.csv':
                frame = pd.read_csv(p, dtype=str, keep_default_na=False)
                for column in ['puzzle_id', 'PuzzleId']:
                    if column in frame:
                        for value in frame[column]: add_id(value)
                for column in ['solver_fen']:
                    if column in frame:
                        positions.update(position_key(v) for v in frame[column] if '/' in v)
            elif p.suffix == '.jsonl' and p.name in ['outcomes.jsonl', 'requests.jsonl']:
                for row in records(p):
                    add_id(row.get('puzzle_id'))
                    body = row.get('request', {})
                    text = json.dumps(body)
                    for match in re.findall(r'FEN: ([rnbqkpRNBQKP1-8/]+ [wb] [KQkq-]+ [a-h1-8-]+)', text):
                        positions.add(position_key(match))
            elif p.suffix == '.json' and any(s in p.name for s in ['sampl', 'manifest', 'reserv']):
                scan_metadata(json.loads(p.read_text(encoding='utf-8')))
            else: continue
            if len(ids) != before or p.name in ['sampling.json', 'pilot_sampling_metadata.json']:
                sources.append({'path': p.relative_to(ROOT).as_posix(), 'sha256': sha(p)})
    return ids, positions, sources

sample_file = OUT/'puzzles.csv'
if sample_file.exists():
    sampling = json.loads((OUT/'sampling.json').read_text())
    assert sha(sample_file) == sampling['sample_sha256']
    puzzles = pd.read_csv(sample_file, dtype={'puzzle_id': str})
else:
    excluded, old_positions, exclusion_sources = historical_exclusions()
    save(OUT/'exclusions.json', {'puzzle_ids': sorted(excluded), 'position_keys': sorted(old_positions), 'sources': exclusion_sources})
    rng = np.random.default_rng(CONFIG['sample_seed'])
    cells = {(s['rating'], s['length']): s for s in CONFIG['cell_sizes']}
    picks = {key: pd.DataFrame() for key in cells}
    feature_path = ROOT/'data/validation/full/validated_features.csv'
    source_path = ROOT/'data/lichess_db_puzzle.csv.zst'
    scanned = 0
    for chunk in pd.read_csv(feature_path, chunksize=100_000, dtype={'puzzle_id': str}):
        frame = chunk.loc[~chunk.puzzle_id.isin(excluded)].copy()
        frame['_position_key'] = frame.solver_fen.map(position_key)
        frame = frame.loc[~frame._position_key.isin(old_positions)].copy()
        frame['rating_stratum'] = np.where(frame.puzzle_rating <= 1157, 'low', np.where(frame.puzzle_rating <= 1700, 'middle', 'high'))
        frame['length_group'] = np.where(frame.total_solver_moves.eq(1), 'one_move', 'multi_move')
        frame['_priority'] = rng.random(len(frame))
        for key, spec in cells.items():
            group = frame.loc[frame.rating_stratum.eq(key[0]) & frame.length_group.eq(key[1])]
            capacity = spec['n']+128
            picks[key] = pd.concat([picks[key], group.nsmallest(capacity, '_priority')], ignore_index=True).nsmallest(capacity, '_priority')
        scanned += len(chunk)
        if scanned % 1_000_000 == 0: print('I scanned', f'{scanned:,}', 'validated positions.', flush=True)
    seen, selected = set(old_positions), []
    for key, spec in cells.items():
        rows = []
        for row in picks[key].sort_values(['_priority', 'puzzle_id']).to_dict('records'):
            if row['_position_key'] in seen: continue
            seen.add(row['_position_key']); rows.append(row)
            if len(rows) == spec['n']: break
        assert len(rows) == spec['n'], 'I need a larger reservoir in a new version; no sample was frozen.'
        selected.append(pd.DataFrame(rows))
    puzzles = pd.concat(selected, ignore_index=True)
    wanted = set(puzzles.puzzle_id); assert len(wanted) == 1000 and not wanted & excluded
    found = []
    for chunk in pd.read_csv(source_path, compression='zstd', chunksize=100_000, dtype=str,
                             keep_default_na=False, usecols=['PuzzleId', 'FEN', 'Moves', 'GameUrl']):
        matched = chunk.loc[chunk.PuzzleId.isin(wanted)]
        if not matched.empty: found.append(matched)
    source = pd.concat(found).rename(columns={'PuzzleId': 'puzzle_id', 'FEN': 'original_fen', 'Moves': 'official_moves', 'GameUrl': 'game_url'})
    puzzles = puzzles.merge(source, on='puzzle_id', validate='one_to_one')
    puzzles['official_first_move'] = puzzles.official_moves.str.split().str[1]
    puzzles['random_success_probability'] = 1/puzzles.legal_moves
    puzzles['split'] = 'train'
    split_rng = np.random.default_rng(CONFIG['split_seed'])
    for key, spec in cells.items():
        indices = puzzles.index[puzzles.rating_stratum.eq(key[0]) & puzzles.length_group.eq(key[1])].to_numpy()
        puzzles.loc[split_rng.permutation(indices)[:spec['test_n']], 'split'] = 'test'
    # I place a rating/length-balanced training prefix first for the operational check.
    order_rng = np.random.default_rng(CONFIG['order_seed'])
    queues = [list(order_rng.permutation(puzzles.index[puzzles.split.eq('train') & puzzles.rating_stratum.eq(k[0]) & puzzles.length_group.eq(k[1])])) for k in cells]
    order = []
    while any(queues):
        for queue in queues:
            if queue: order.append(queue.pop())
    order += list(order_rng.permutation(puzzles.index[puzzles.split.eq('test')]))
    puzzles = puzzles.loc[order].reset_index(drop=True)
    puzzles['collection_order'] = np.arange(1, 1001)
    puzzles['operational_check'] = puzzles.collection_order.le(CONFIG['warmup_n'])
    puzzles = puzzles.drop(columns=['_priority', '_position_key'])
    for row in puzzles.itertuples(index=False):
        board = chess.Board(row.original_fen)
        for i, move in enumerate(row.official_moves.split()):
            assert chess.Move.from_uci(move) in board.legal_moves
            board.push_uci(move)
            if i == 0:
                assert board.fen() == row.solver_fen and board.legal_moves.count() == row.legal_moves
        assert chess.Board(row.solver_fen).is_valid()
    puzzles.to_csv(sample_file, index=False)
    sampling = {'n': 1000, 'train_n': 800, 'test_n': 200, 'warmup_n': 20,
                'sample_seed': CONFIG['sample_seed'], 'split_seed': CONFIG['split_seed'], 'order_seed': CONFIG['order_seed'],
                'excluded_id_count': len(excluded), 'excluded_position_count': len(old_positions),
                'sample_sha256': sha(sample_file), 'source_sha256': sha(source_path), 'features_sha256': sha(feature_path),
                'exclusions_sha256': sha(OUT/'exclusions.json'),
                'scope': 'Approximately equal allocation to six strata; accuracy describes this balanced sample, not natural puzzle frequencies.'}
    save(OUT/'sampling.json', sampling)
    save(OUT/'environment.json', {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                                'chess': chess.__version__, 'created_utc': datetime.now(timezone.utc).isoformat()})
assert len(puzzles) == puzzles.puzzle_id.nunique() == puzzles.solver_fen.map(position_key).nunique() == 1000
assert puzzles.split.value_counts().to_dict() == {'train': 800, 'test': 200}
assert puzzles.loc[puzzles.operational_check, 'split'].eq('train').all()
assert not set(puzzles.puzzle_id) & set(json.loads((OUT/'exclusions.json').read_text())['puzzle_ids'])
assert not set(puzzles.solver_fen.map(position_key)) & set(json.loads((OUT/'exclusions.json').read_text())['position_keys'])
display(puzzles.groupby(['rating_stratum', 'length_group', 'split']).size().unstack('split').reindex(['low','middle','high'], level=0))
print('I froze', len(puzzles), 'distinct puzzle positions and excluded', sampling['excluded_id_count'], 'earlier/reserved IDs.')


split                        test  train
rating_stratum length_group             
low            multi_move      33    134
               one_move        34    133
middle         multi_move      33    133
               one_move        33    133
high           multi_move      34    133
               one_move        33    134

I froze 1000 distinct puzzle positions and excluded 282 earlier/reserved IDs.


## 3. I collect with saved responses and independent provider quotas

In [4]:

def epoch(row):
    if row.get('finished_epoch') is not None: return float(row['finished_epoch'])
    return datetime.fromisoformat(row['timestamp_utc'].replace('Z', '+00:00')).timestamp()

def log_model(row):
    return row.get('model') or row.get('request', {}).get('model') or row.get('response', {}).get('modelVersion') or row.get('response', {}).get('model')

def usage_tokens(spec, row):
    raw = row.get('response', {})
    usage = raw.get('usage', {}) if spec['provider'] == 'groq' else raw.get('usageMetadata', {})
    return int(usage.get('total_tokens', usage.get('totalTokenCount', 0)) or 0)

def next_pacific_reset(now):
    local = datetime.fromtimestamp(now, PACIFIC)
    return datetime.combine(local.date()+timedelta(days=1), datetime.min.time(), PACIFIC).timestamp()+5

class Collector:
    def __init__(self, name, spec):
        self.name, self.spec = name, spec
        self.folder = OUT/name; self.folder.mkdir(exist_ok=True)
        self.log_path = self.folder/'requests.jsonl'; self.outcome_path = self.folder/'outcomes.jsonl'
        self.session = requests.Session()
        self.old_logs = []
        for path in (ROOT/'data/experiments').rglob('requests.jsonl'):
            if OUT in path.parents: continue
            self.old_logs.extend(r for r in records(path) if log_model(r) == spec['model'])
    def connect(self):
        key = (ROOT/self.spec['key_file']).read_text(encoding='utf-8-sig').strip()
        if self.spec['provider'] == 'groq':
            self.session.headers['Authorization'] = 'Bearer '+key; url = self.spec['base_url']+'/models'
        else:
            self.session.headers['x-goog-api-key'] = key; url = self.spec['base_url']+'/models/'+self.spec['model']
        del key
        try:
            response = self.session.get(url, timeout=30)
            available = response.ok and (self.spec['provider'] == 'google' or self.spec['model'] in {m['id'] for m in response.json().get('data', [])})
            save(self.folder/'access.json', {'available': bool(available), 'http_status': response.status_code})
            if available: save(self.folder/'provider_metadata.json', response.json())
            return bool(available)
        except (requests.RequestException, ValueError) as error:
            save(self.folder/'access.json', {'available': False, 'error_type': type(error).__name__}); return False
    def latest(self): return {r['case_id']: r for r in records(self.outcome_path)}
    def history(self):
        merged = {}
        for row in self.old_logs+records(self.log_path):
            merged[(row.get('trial_id'), row.get('retry'), row.get('timestamp_utc'))] = row
        return list(merged.values())
    def quota_wait(self, body, now=None):
        now = time.time() if now is None else now; spec = self.spec; history = self.history()
        state_path = self.folder/'pause.json'; pause = json.loads(state_path.read_text()) if state_path.exists() else {}
        if pause.get('requires_review'): return 0, pause.get('reason', 'Access review required'), True
        wait = max(0, float(pause.get('resume_epoch', 0))-now); reason = 'Saved provider retry time' if wait else None
        if history:
            last = max(history, key=epoch)
            pacing = max(0, spec['minimum_pause_seconds']-(now-epoch(last)))
            if pacing > wait: wait, reason = pacing, 'Request pacing'
        if spec['provider'] == 'google':
            day = datetime.fromtimestamp(now, PACIFIC).date()
            count = sum(datetime.fromtimestamp(epoch(r), PACIFIC).date() == day for r in history)
            if count >= spec['daily_request_budget']:
                reset = next_pacific_reset(now)-now
                if reset > wait: wait, reason = reset, 'Conservative project-day request budget'
        else:
            recent = sorted([r for r in history if epoch(r) > now-86400], key=epoch)
            estimate = math.ceil(len(body['messages'][0]['content'])/2.5)+128
            reservation = spec['max_completion_tokens']+estimate
            spent = sum(usage_tokens(spec, r) or (reservation if r.get('http_status') in [None,500,502,503,504] else 0) for r in recent)
            count = len(recent)
            if spent+reservation > spec['daily_token_budget'] or count >= spec['rolling_request_budget']:
                remaining_spent, remaining_count = spent, count
                for row in recent:
                    remaining_spent -= usage_tokens(spec, row) or (reservation if row.get('http_status') in [None,500,502,503,504] else 0)
                    remaining_count -= 1
                    if remaining_spent+reservation <= spec['daily_token_budget'] and remaining_count < spec['rolling_request_budget']:
                        reset = epoch(row)+86400-now+5
                        if reset > wait: wait, reason = reset, 'Conservative rolling 24-hour budget'
                        break
            if history:
                last = max(history, key=epoch); headers = last.get('rate_limits', {})
                allowance = float(headers.get('x-ratelimit-limit-tokens', 8000))
                if reservation > allowance: return 0, 'Request reservation exceeds per-minute allowance', True
                if float(headers.get('x-ratelimit-remaining-tokens', -1)) < reservation:
                    reset = max(0, (seconds(headers.get('x-ratelimit-reset-tokens')) or 60)-(now-epoch(last))+2)
                    if reset > wait: wait, reason = reset, 'Groq token reservation'
        return wait, reason, False
    def defer(self, reason, wait=0, review=False):
        row = {'status': 'needs_review' if review else 'paused_for_quota', 'reason': reason,
               'requires_review': review, 'resume_epoch': time.time()+wait,
               'resume_utc': datetime.fromtimestamp(time.time()+wait, timezone.utc).isoformat()}
        save(self.folder/'pause.json', row)
        return row
    def completed_row(self, log, board, task, expected):
        raw = log['response']; value = normalized(self.spec, raw)
        if not value.get('choices'):
            value = {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
        category, answer, usable = score(board, task, value, expected)
        row = {'case_id': log['case_id'], 'puzzle_id': log['puzzle_id'], 'task': task, 'model_key': self.name,
               'trial_id': log['trial_id'], 'status': 'completed', 'failure_type': category,
               'answer': answer, 'usable_answer': bool(usable), 'success': int(category == 'success'),
               'returned_model': raw.get('model', raw.get('modelVersion')), 'total_tokens': usage_tokens(self.spec, log)}
        append(self.outcome_path, row)
        return row
    def run_case(self, puzzle_id, board, task='evaluation', expected=None):
        case_id = task+':'+puzzle_id; body = request_body(self.spec, board, task)
        cached = self.latest().get(case_id)
        if cached and cached['status'] == 'completed': return cached
        # I replay a durably saved response if a crash happened before its outcome was written.
        successful = [r for r in records(self.log_path) if r['case_id'] == case_id and 'response' in r]
        if successful:
            log = successful[-1]; assert log['request'] == body
            return self.completed_row(log, board, task, expected)
        trial = str(uuid.uuid4())
        for retry in range(5):
            wait, reason, review = self.quota_wait(body)
            if review: return self.defer(reason, review=True)
            if wait > 120: return self.defer(reason, wait)
            while wait > 0:
                time.sleep(min(30, wait)); wait, reason, review = self.quota_wait(body)
                if review: return self.defer(reason, review=True)
                if wait > 120: return self.defer(reason, wait)
            spec = self.spec; http = None; retry_wait = None
            url = spec['base_url']+('/chat/completions' if spec['provider'] == 'groq' else '/models/'+spec['model']+':generateContent')
            log = {'trial_id': trial, 'case_id': case_id, 'puzzle_id': puzzle_id, 'task': task,
                   'model': spec['model'], 'provider': spec['provider'], 'retry': retry,
                   'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'request': body}
            try:
                response = self.session.post(url, json=body, timeout=(15,240)); http = response.status_code
                log['http_status'] = http
                log['rate_limits'] = {k.lower(): v for k, v in response.headers.items() if k.lower().startswith(('x-ratelimit','retry-after'))}
                if response.ok:
                    raw = response.json(); assert isinstance(raw, dict)
                    log['response'] = raw
                else:
                    try: log['provider_error'] = response.json().get('error', {})
                    except ValueError: log['provider_error'] = {'type': 'non_json_error'}
                    retry_wait = seconds(response.headers.get('retry-after')) or 60
            except (requests.RequestException, ValueError, AssertionError) as error:
                log['error_type'] = type(error).__name__
            log['finished_epoch'] = time.time(); append(self.log_path, log)
            if 'response' in log:
                row = self.completed_row(log, board, task, expected)
                earlier_models = {r.get('returned_model') for r in self.latest().values() if r['case_id'] != case_id and r.get('returned_model')}
                if earlier_models and row['returned_model'] not in earlier_models:
                    self.defer('Returned model version changed during collection', review=True)
                return row
            if http == 429:
                error = log.get('provider_error', {}); message = json.dumps(error).lower()
                zero = any(str(v) == '0' for k, v in log['rate_limits'].items() if 'limit' in k and 'remaining' not in k)
                violations = [v for d in error.get('details', []) for v in d.get('violations', [])] if isinstance(error, dict) else []
                zero = zero or any(str(v.get('quotaValue')) == '0' for v in violations)
                if zero or 'request too large' in message:
                    return self.defer('Provider reported zero allowance or request-size restriction; inspect raw error', review=True)
                daily = any(v in message for v in ['perday','per_day','per day','tokens per day','daily','monthly'])
                for detail in error.get('details', []) if isinstance(error, dict) else []:
                    if 'retryDelay' in detail: retry_wait = seconds(detail['retryDelay']) or retry_wait
                if daily:
                    reset = next_pacific_reset(time.time())-time.time() if spec['provider'] == 'google' else 86400
                    return self.defer('Provider daily/monthly quota; account check may be needed', max(retry_wait, reset))
                if retry_wait > 120 or retry == 4: return self.defer('Provider rate limit', retry_wait+2)
                self.defer('Provider retry-after', retry_wait+2)
            elif http not in [None,200,500,502,503,504]:
                return self.defer(f'Provider access/configuration error HTTP {http}', review=True)
            elif retry == 4: return self.defer('Repeated transport/server failure; inspect logs', review=True)
            else:
                time.sleep(min(30, 5*2**retry))
        return self.defer('Unexpected retry exhaustion', review=True)


## 4. I verify reliability before live requests

In [5]:

def audit_model(collector):
    spec = collector.spec
    cases = {r.puzzle_id: (chess.Board(r.solver_fen), 'evaluation', r.official_first_move) for r in puzzles.itertuples(index=False)}
    cases.update({p: (b, t, None) for p, b, t in FIXTURES})
    logs = [r for r in records(collector.log_path) if 'response' in r]
    by_trial = {(r['trial_id'], r['case_id']): r for r in logs}
    latest = collector.latest(); checked = 0
    assert len({r['case_id'] for r in logs}) == len(logs), 'More than one saved successful response for a case'
    for row in latest.values():
        if row['status'] != 'completed': continue
        board, task, expected = cases[row['puzzle_id']]
        log = by_trial[(row['trial_id'], row['case_id'])]
        assert log['request'] == request_body(spec, board, task)
        value = normalized(spec, log['response'])
        if not value.get('choices'): value = {'choices': [{'message': {'content': ''}, 'finish_reason': 'blocked_or_empty'}]}
        category, answer, usable = score(board, task, value, expected)
        assert (category, answer, usable, int(category == 'success')) == (row['failure_type'], row['answer'], row['usable_answer'], row['success'])
        checked += 1
    assert checked == len(logs), 'An unanswered log must be recovered before analysis'
    versions = sorted({r['returned_model'] for r in latest.values() if r.get('returned_model')})
    assert len(versions) <= 1, 'Provider returned different model versions; review before analysis'
    evaluated = [r for r in latest.values() if r['task'] == 'evaluation' and r['status'] == 'completed']
    if evaluated:
        frame = puzzles.merge(pd.DataFrame(evaluated), on='puzzle_id', how='left', validate='one_to_one')
    else:
        frame = puzzles.copy()
        for column in ['case_id','trial_id','status','failure_type','answer','usable_answer','success','returned_model','total_tokens']: frame[column] = np.nan
    frame.to_csv(collector.folder/'results.csv', index=False)
    audit = {'passed': True, 'responses_verified': checked, 'research_responses': len(evaluated),
             'sample_sha256': sha(sample_file), 'results_sha256': sha(collector.folder/'results.csv'),
             'returned_models': versions, 'manifest_sha256': sha(OUT/'manifest.json')}
    save(collector.folder/'audit.json', audit)
    return frame, audit

def provider_once(collector):
    proof_path = collector.folder/'resume_check.json'
    proof = json.loads(proof_path.read_text()) if proof_path.exists() else {}
    prefix_ids = list(puzzles.loc[puzzles.operational_check, 'puzzle_id'])
    latest = collector.latest()
    prefix_verified = proof.get('new_generation_requests_on_resume') == 0 and all(latest.get('evaluation:'+p, {}).get('status') == 'completed' for p in prefix_ids)
    capabilities = []; failure_streak = 0
    for puzzle_id, board, task in FIXTURES:
        row = collector.run_case(puzzle_id, board, task)
        capabilities.append({'puzzle_id': puzzle_id, 'task': task, **row})
        if row['status'] != 'completed': break
        failure_streak = failure_streak+1 if row['failure_type'] == 'generation_limit' else 0
        if failure_streak >= 2: break
    counts = {task: sum(r.get('success',0) == 1 and r['task'] == task for r in capabilities) for task in ['side','legal','mate']}
    passed = len(capabilities) == 8 and all(r['status'] == 'completed' and r.get('usable_answer') for r in capabilities) and counts['side'] == 2 and counts['legal'] == 2 and counts['mate'] >= 2
    gate = {'passed': bool(passed), 'n': len(capabilities), 'successes': counts}
    save(collector.folder/'capability_gate.json', gate)
    pd.DataFrame(capabilities).to_csv(collector.folder/'capability.csv', index=False)
    stop = None
    if passed:
        selected = puzzles if RUN_MODE == 'bulk' and prefix_verified else puzzles.loc[puzzles.operational_check]
        for row in selected.itertuples(index=False):
            previous = collector.latest().get('evaluation:'+row.puzzle_id)
            result = collector.run_case(row.puzzle_id, chess.Board(row.solver_fen), expected=row.official_first_move)
            if result['status'] != 'completed': stop = result; break
            if previous is None: print(collector.name, 'saved', row.collection_order, '/1000', flush=True)
    elif capabilities and capabilities[-1]['status'] != 'completed': stop = capabilities[-1]
    frame, audit = audit_model(collector)
    research_n = int(frame.status.eq('completed').sum())
    warmup_complete = bool(frame.loc[frame.operational_check, 'status'].eq('completed').all())
    if passed and warmup_complete and not prefix_verified:
        before = {'requests': sha(collector.log_path), 'outcomes': sha(collector.outcome_path)}
        for row in puzzles.loc[puzzles.operational_check].itertuples(index=False):
            collector.run_case(row.puzzle_id, chess.Board(row.solver_fen), expected=row.official_first_move)
        assert before == {'requests': sha(collector.log_path), 'outcomes': sha(collector.outcome_path)}, 'Resume changed completed answers or issued new generation requests'
        save(proof_path, {'model': collector.name, 'training_cases': 20, 'new_generation_requests_on_resume': 0,
                         'requests_unchanged_on_resume': True, 'outcomes_unchanged_on_resume': True,
                         'test_cases_used': 0, 'prefix_ids': prefix_ids, 'hashes': before})
        prefix_verified = True
    summary = {'model_key': collector.name, 'label': collector.spec['label'], 'capability_passed': passed,
               'research_n': research_n, 'planned_n': 1000, 'training_n': int((frame.status.eq('completed') & frame.split.eq('train')).sum()),
               'test_n': int((frame.status.eq('completed') & frame.split.eq('test')).sum()),
               'warmup_complete': warmup_complete, 'prefix_verified': bool(prefix_verified), 'complete': research_n == 1000,
               'status': 'complete' if research_n == 1000 else stop.get('status') if stop else 'operational_check_complete' if warmup_complete else 'capability_gate_failed',
               'pause': stop, 'audit': audit}
    save(collector.folder/'summary.json', summary)
    return summary

def run_provider(name, spec):
    collector = Collector(name, spec)
    try:
        with provider_lock(collector.folder):
            if not collector.connect(): return {'model_key': name, 'status': 'access_blocked', 'research_n': 0, 'warmup_complete': False}
            while True:
                result = provider_once(collector)
                if RUN_MODE == 'bulk' and result['status'] == 'operational_check_complete' and result['prefix_verified']:
                    continue  # I verified this provider's prefix; I can now advance it independently.
                if RUN_MODE != 'bulk' or not WAIT_FOR_QUOTA or result['complete'] or result['status'] != 'paused_for_quota': return result
                pause = result['pause']
                print(name, 'paused; next eligible UTC time:', pause.get('resume_utc'), flush=True)
                while time.time() < pause['resume_epoch']: time.sleep(max(0, min(30, pause['resume_epoch']-time.time())))
    finally: collector.session.close()


def offline_checks():
    # I test quota/immutability/crash paths with fake HTTP responses, never paid/live calls.
    import tempfile
    class FakeResponse:
        def __init__(self, status, body, headers=None): self.status_code, self.body, self.headers = status, body, headers or {}; self.ok = status == 200
        def json(self): return self.body
    class FakeSession:
        def __init__(self, responses): self.responses, self.calls = responses, 0
        def post(self, *args, **kwargs): self.calls += 1; return self.responses.pop(0)
        def close(self): pass
    with tempfile.TemporaryDirectory(prefix='offline_check_', dir=OUT) as temporary:
        assert Path(temporary).resolve().is_relative_to(OUT.resolve())
        def mock(label, name='gpt_oss'):
            c = Collector(name, dict(CONFIG['models'][name])); c.session.close()
            c.folder = Path(temporary)/label; c.folder.mkdir(); c.log_path = c.folder/'requests.jsonl'; c.outcome_path = c.folder/'outcomes.jsonl'
            c.old_logs = []; c.spec['minimum_pause_seconds'] = 0
            return c
        fake = {'choices': [{'message': {'content': 'e2e4'}, 'finish_reason': 'stop'}], 'model': 'openai/gpt-oss-120b', 'usage': {'total_tokens': 500}}
        c = mock('saved'); c.session = FakeSession([FakeResponse(200, fake)])
        first = c.run_case('test1', start, expected='d2d4'); assert first['success'] == 0 and first['failure_type'] == 'legal_incorrect'
        before = sha(c.outcome_path); assert c.run_case('test1', start, expected='d2d4') == first
        assert c.session.calls == 1 and sha(c.outcome_path) == before
        c.outcome_path.unlink(); recovered = c.run_case('test1', start, expected='d2d4')
        assert recovered == first and c.session.calls == 1 and sha(c.outcome_path) == before
        c = mock('zero'); c.session = FakeSession([FakeResponse(429, {'error': {'message': 'Rate limited'}}, {'x-ratelimit-limit-tokens': '0'})])
        result = c.run_case('zero1', start, expected='e2e4')
        assert result['status'] == 'needs_review' and c.session.calls == 1 and not c.latest()
        c = mock('daily', 'gemini'); c.spec['daily_request_budget'] = 1
        now = time.time(); c.old_logs = [{'trial_id': 'q1', 'retry': 0, 'timestamp_utc': datetime.now(timezone.utc).isoformat(), 'finished_epoch': now, 'http_status': 200}]
        wait, reason, review = c.quota_wait(request_body(c.spec, start), now)
        assert wait > 0 and not review and 'project-day' in reason
        c = mock('tokens'); c.old_logs = [{'trial_id': 'q2', 'retry': 0, 'finished_epoch': time.time(), 'http_status': 200, 'response': {'usage': {'total_tokens': 190000}}}]
        wait, reason, review = c.quota_wait(request_body(c.spec, start)); assert wait > 86000 and not review
    fake = lambda text, finish='stop': {'choices': [{'message': {'content': text}, 'finish_reason': finish}]}
    assert score(start, 'legal', fake('e4'))[1] == 'e2e4'
    assert score(start, 'legal', fake('Move: e2e4'))[0] == 'unparseable'
    assert score(start, 'legal', fake('e2e5'))[0] == 'illegal_move'
    assert score(start, 'legal', fake('e2e4','length'))[0] == 'generation_limit'
    assert score(mate, 'mate', fake('g6g7'))[0] == 'success'
    print('My offline checks passed: immutable wrong answers, crash replay without new calls, zero quota, daily/rolling budgets and scoring.')

offline_checks()


My offline checks passed: immutable wrong answers, crash replay without new calls, zero quota, daily/rolling budgets and scoring.


## 5. I run the selected collection mode and verify resume

In [6]:

summary = {}
if RUN_MODE == 'setup':
    print('My sample, split, protocol and offline checks are ready. I have sent no API requests in setup mode.')
else:
    # Each provider enforces its own capability, training-prefix and immutable-resume checks.
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = {pool.submit(run_provider, name, spec): name for name, spec in CONFIG['models'].items()}
        for future in as_completed(futures): summary[futures[future]] = future.result()
    save(OUT/'progress.json', summary)
    if RUN_MODE == 'warmup':
        # I rerun the exact same prefix to prove that resume issues no new generation requests.
        before = {name: {'requests': sha(OUT/name/'requests.jsonl'), 'outcomes': sha(OUT/name/'outcomes.jsonl')}
                  for name in CONFIG['models'] if (OUT/name/'outcomes.jsonl').exists()}
        ready = len(before) == 2 and all(s.get('warmup_complete') and s.get('audit', {}).get('passed') for s in summary.values())
        if ready:
            for name, spec in CONFIG['models'].items(): run_provider(name, spec)
            unchanged = all(before[name]['requests'] == sha(OUT/name/'requests.jsonl') and before[name]['outcomes'] == sha(OUT/name/'outcomes.jsonl') for name in before)
            assert unchanged, 'Resume changed a completed response or issued a new generation request.'
        else: unchanged = False
        save(OUT/'warmup_validation.json', {'ready': bool(ready and unchanged), 'planned_training_cases_per_model': 20,
             'completed_answers_unchanged_on_resume': unchanged, 'new_generation_requests_on_resume': 0 if unchanged else None,
             'test_cases_used': 0, 'scope': 'Operational readiness only; accuracy does not determine whether I continue.',
             'before_request_outcome_hashes': before})
    display(pd.DataFrame([{'Model': CONFIG['models'][name]['label'], 'Status': value['status'],
                          'Saved research answers': value['research_n'], 'Planned': 1000,
                          'Operational check complete': value.get('warmup_complete', False)} for name, value in summary.items()]))
assert all(sha(ROOT/path) == digest for path, digest in protected.items())
print('I preserved historical response/model files. All progress is saved in data/experiments/bulk_v1/.')


I started full collection and paused at local usage guards.
Gemini: 404/1,000 research answers; GPT-OSS: 0/1,000.
Both saved audits passed. My held-out test puzzles remain unused.
Detailed progress and raw responses are saved in data/experiments/bulk_v1/.


,Model,Status,Saved research answers,Planned,Operational check complete
0,GPT-OSS 120B,paused_for_quota,0,1000,False
1,Gemini 3.5 Flash-Lite,paused_for_quota,404,1000,True


## How I continue

I leave `RUN_MODE="bulk"` and rerun this notebook after the saved pause time. Each provider resumes independently. It verifies its own 20-case training prefix before advancing. With `WAIT_FOR_QUOTA=False`, I return at each provider's first limit instead of waiting overnight.

I use notebook 05 to create `early_results_2` from available training answers only. I use notebook 06 after full audited collection for the prespecified association regressions and held-out tree evaluation. I do not change predictors or settings based on the interim graphs. I back up my Git-ignored data separately.

## My saved collection checkpoint

I paused at 404 Gemini research answers under my conservative project-day request guard. GPT-OSS paused before generation at its saved local rolling-budget retry time. Neither stop was a new provider rate-limit error. My earliest saved retry times are October 22, 2026, 1:17 AM New York for GPT-OSS and 3:00 AM New York for Gemini, subject to another usage check.

I generated my separate training-only graphs in notebook 05, `early_results_2`. I resume this notebook with `RUN_MODE = "bulk"`; completed answers are retained and audited rather than requested again. I leave my final analysis in notebook 06 until each model has all 1,000 audited answers.